# Retrieval

<img src="./assets/LC_retrieval.png">

大型语言模型（LLMs）虽然功能强大，但存在两个主要局限性：

- 上下文长度有限——无法一次性处理整个语料库。
- 知识静态——训练数据被冻结在某个时间点后就不再更新。

检索（Retrieval） 通过在查询时获取相关的外部知识来解决这些问题。这是 检索增强生成（RAG, Retrieval-Augmented Generation） 的基础：用特定上下文信息增强 LLM 的回答。

## 构建知识库

知识库是在检索过程中使用的文档或结构化数据的存储库。

- 如果你需要自定义知识库，可以使用 LangChain 的 **文档加载器（Document Loaders）** 和 **向量存储（Vector Stores）** 从自有数据构建。
- 如果你已有知识库（例如 SQL 数据库、CRM 系统或内部文档系统），则无需重建。你可以：
  - 在 **Agentic RAG** 中将其作为智能体的工具连接；
  - 查询它并将检索到的内容作为上下文提供给 LLM（即 **两步 RAG**）。

👉 教程：[语义搜索](https://docs.langchain.com/oss/python/langchain/knowledge-base) —— 学习如何使用 LangChain 的文档加载器、嵌入模型和向量存储，从你自己的数据构建可搜索的知识库。

### 从检索到 RAG

检索使 LLM 能够在运行时访问相关上下文。但大多数实际应用更进一步：将检索与生成结合，以生成有依据、上下文感知的回答。这就是 **检索增强生成（RAG）** 的核心思想。检索流水线成为结合搜索与生成的更广泛系统的基础。

### Retrieval Pipeline

典型的检索工作流如下：

1. **文档加载** → 2. **文本分割** → 3. **嵌入生成** → 4. **存入向量数据库** → 5. **查询检索** → 6. **LLM使用检索到的信息** → 7. **生成回答**

每个组件都是模块化的：你可以更换加载器、分割器、嵌入模型或向量存储，而无需重写应用逻辑。

### 构建模块

| 组件 | 说明 |
|------|------|
| **文档加载器（Document Loaders）** | 从外部源（如 Google Drive、Slack、Notion 等）导入数据，返回标准化的 `Document` 对象。 |
| **文本分割器（Text Splitters）** | 将大文档拆分为小片段，便于单独检索并适应模型上下文窗口。 |
| **嵌入模型（Embedding Models）** | 将文本转换为向量，使语义相近的文本在向量空间中彼此靠近。 |
| **向量存储（Vector Stores）** | 用于存储和搜索嵌入向量的专用数据库。 |
| **检索器（Retrievers）** | 接收非结构化查询并返回相关文档的接口。 |

## RAG 架构类型

| 架构 | 描述 | 控制性 | 灵活性 | 延迟 | 典型用例 |
|------|------|--------|--------|------|----------|
| **两步 RAG（2-Step RAG）** | 总是先检索再生成，简单且可预测 | 高 | 低 | 快 | FAQ、文档问答机器人 |
| **Agentic RAG** | 由 LLM 驱动的智能体在推理过程中决定何时以及如何检索 | 低 | 高 | 可变 | 需要多工具的研究助手 |
| **混合 RAG（Hybrid RAG）** | 结合两者特性，加入验证步骤 | 中 | 中 | 可变 | 需质量控制的领域问答 |

> 💡 **延迟说明**：两步 RAG 的延迟通常更可预测（LLM 调用次数固定），但实际延迟还受 API 响应、网络或数据库性能影响。

### 2-Step RAG

检索步骤总是在生成步骤之前执行。这种架构简单、可预测，适用于检索是生成前提的场景（如客服问答、技术文档查询）。

## Agentic RAG

Agentic RAG 将检索增强生成与基于智能体的推理相结合。智能体（由 LLM 驱动）会逐步推理，并在交互过程中动态决定何时、如何检索信息。
只需为智能体提供一个或多个能获取外部知识的工具（如文档加载器、Web API、数据库查询等），即可启用 RAG 行为。

<img src="./assets/LC_agent_rag.png">

本示例实现了一个基于代理的 RAG 系统，用于辅助用户查询 LangGraph 文档。代理首先加载llms.txt 文件（其中列出了可用的文档 URL），然后可以根据用户的问题动态地使用工具 `fetch_documentation` 来检索和处理相关内容。

In [1]:
import requests
import os
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from markdownify import markdownify
from langchain_openai import ChatOpenAI


ALLOWED_DOMAINS = ["https://langchain-ai.github.io/"]
LLMS_TXT = 'https://langchain-ai.github.io/langgraph/llms.txt'


@tool
def fetch_documentation(url: str) -> str:  
    """Fetch and convert documentation from a URL"""
    if not any(url.startswith(domain) for domain in ALLOWED_DOMAINS):
        return (
            "Error: URL not allowed. "
            f"Must start with one of: {', '.join(ALLOWED_DOMAINS)}"
        )
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return markdownify(response.text)


# We will fetch the content of llms.txt, so this can
# be done ahead of time without requiring an LLM request.
llms_txt_content = requests.get(LLMS_TXT).text

# System prompt for the agent
system_prompt = f"""
You are an expert Python developer and technical assistant.
Your primary role is to help users with questions about LangGraph and related tools.

Instructions:

1. If a user asks a question you're unsure about — or one that likely involves API usage,
   behavior, or configuration — you MUST use the `fetch_documentation` tool to consult the relevant docs.
2. When citing documentation, summarize clearly and include relevant context from the content.
3. Do not use any URLs outside of the allowed domain.
4. If a documentation fetch fails, tell the user and proceed with your best expert understanding.

You can access official documentation from the following approved sources:

{llms_txt_content}

You MUST consult the documentation to get up to date documentation
before answering a user's question about LangGraph.

Your answers should be clear, concise, and technically accurate.
"""

tools = [fetch_documentation]

model = ChatOpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-max",
    max_tokens=3_000
)

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    name="Agentic RAG",
)

for step in agent.stream({
    'messages': [
        HumanMessage(content=(
            "Write a short example of a langgraph agent using the "
            "prebuilt create react agent. the agent should be able "
            "to look up stock pricing information."
        ))
    ]
}, stream_mode="values"):
    step["messages"][-1].pretty_print()


# print(response['messages'][-1].content)

================================ Human Message =================================

Write a short example of a langgraph agent using the prebuilt create react agent. the agent should be able to look up stock pricing information.
================================== Ai Message ==================================

To create a LangGraph agent that can look up stock pricing information using the prebuilt `create_react_agent`, we'll need to follow these steps:

1. Set up the environment and install necessary dependencies.
2. Define a tool for fetching stock prices.
3. Create the agent using `create_react_agent`.
4. Integrate the tool with the agent.
5. Test the agent.

Let's go through each step in detail.

### Step 1: Set Up the Environment

First, make sure you have the necessary dependencies installed. You can install them using pip:

```bash
pip install langchain
pip install yfinance  # For fetching stock prices
```

### Step 2: Define a Tool for Fetching Stock Prices

We'll use the `yfinanc

### 混合 RAG（Hybrid RAG）

结合两步 RAG 与 Agentic RAG 的特点，引入中间步骤以提升质量：

- **查询增强**：重写模糊问题、生成多个查询变体。
- **检索验证**：判断检索结果是否相关充分，否则重新检索。
- **答案验证**：检查生成答案的准确性，必要时修正。

适用于：
- 查询模糊或不明确的应用
- 需要质量控制的系统
- 多源数据或需迭代优化的工作流

👉 教程：[带自校正的 Agentic RAG](https://docs.langchain.com/oss/python/langgraph/agentic-rag)